# 🌌 Universal Astrological OKF Engine
This notebook serves as a multi-tenant engine for calculating high-precision astrological charts and converting them into the **Open Knowledge Format (OKF)**—a modular, human-readable system designed for knowledge management tools like Obsidian.

In [ ]:
import sys

# Check for Google Colab environment
is_colab = 'google.colab' in sys.modules

missing_packages = []
for pkg_name, import_name in [
    ("pysweph", "swisseph"),
    ("pyyaml", "yaml"),
    ("requests", "requests"),
    ("tzdata", "tzdata"),
    ("jinja2", "jinja2"),
    ("python-dateutil", "dateutil"),
    ("geopy", "geopy")
]:
    try:
        __import__(import_name)
    except ImportError:
        missing_packages.append(pkg_name)

if not missing_packages:
    print("✅ All required packages are installed.")
else:
    print(f"❌ Missing packages: {', '.join(missing_packages)}")
    if is_colab:
        print("Running in Google Colab: Installing missing packages automatically...")
        import subprocess
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
            print("✅ Packages installed successfully. Please restart runtime if needed.")
        except Exception as e:
            print(f"❌ Error installing packages: {e}")
    else:
        print("\nPlease run the following command in your terminal to install them:")
        print(f"    {sys.executable} -m pip install {' '.join(missing_packages)}")
        print("\nAfter installation, please restart the Jupyter kernel in VS Code.")

### 📦 Libraries and Dependencies
This block imports the necessary Python standard libraries and external packages.
- `swisseph` (as `swe`) is the core engine for astronomical calculations.
- `yaml`, `json`, and `hashlib` are used for data formatting and file namespacing.
- `zoneinfo` and `datetime` handle high-precision time conversions.

In [ ]:
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# --- STANDARD LIBRARY IMPORTS ---
import os
import json
import hashlib
import shutil
import types
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo

# --- THIRD-PARTY IMPORTS ---
import requests
import yaml
import swisseph as swe


## 🛰️ Ephemeris Data Acquisition
To ensure scientific-grade precision, we utilize the **Swiss Ephemeris**. This section downloads the necessary planetary data files (`.se1`) from the official repository. These files are required for calculating 'True' planetary positions and node calculations.

In [ ]:
# 1. Create directory
# To run in Google Colab, change this path back to '/content/ephe'
notebook_dir = os.path.abspath('')
ephe_path = os.path.join(notebook_dir, 'ephe')
if not os.path.exists(ephe_path):
    os.makedirs(ephe_path)

# 2. Correct raw URLs
base_url = "https://github.com/aloistr/swisseph/raw/master/ephe/"
urls = [
    base_url + "seas_18.se1",
    base_url + "sepl_18.se1",
    base_url + "semo_18.se1"
]

for url in urls:
    file_name = url.split('/')[-1]
    target = os.path.join(ephe_path, file_name)

    # Cleanup existing corrupt/small files
    if os.path.exists(target) and os.path.getsize(target) < 5000:
        os.remove(target)

    if not os.path.exists(target):
        print(f"Downloading {file_name}...")
        try:
            r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            if r.status_code == 200:
                with open(target, 'wb') as f:
                    f.write(r.content)
                print(f"Successfully downloaded {file_name}")
            else:
                print(f"Failed to download {file_name}: Status {r.status_code}")
        except Exception as e:
            print(f"Error downloading {file_name}: {e}")
    else:
        print(f"✅ {file_name} already exists. Skipping download.")

# 3. Configure and Verify
swe.set_ephe_path(ephe_path)
files_info = {f: f"{os.path.getsize(os.path.join(ephe_path, f)) / 1024:.2f} KB" for f in os.listdir(ephe_path) if f.endswith('.se1')}
print(f"Ephemeris path set to: {ephe_path}")
print(f"Verified Files: {files_info}")


### 🛠️ Configuration and User Inputs
The form below allows you to define the subject's birth data (Date, Time, Location) and specific astrological settings like the **Ayanamsa** (sidereal offset) and **House System**. These variables drive the entire calculation engine.

## ⚙️ Profile Configuration
Use the form below to set the subject's birth data.
- **Subject Name**: Used for namespacing directories and files.
- **Coordinates**: High-precision latitude/longitude is required for accurate calculations.
  * *Automatic Geocoding*: You can set `AUTO_GEOLOCATE = True` and enter the `BIRTH_PLACE` (e.g. `"San Francisco, CA"` or `"Paris, France"`) to automatically retrieve coordinates using `geopy` / OpenStreetMap.
  * *Manual Lookup*: If offline or disabling auto-geolocating, search on [LatLong.net](https://www.latlong.net) or [Google Maps](https://www.google.com/maps) (right-click location to copy coordinates) and input them manually.
- **Settings**: Choose your preferred Ayanamsa (Sidereal Offset) and House System (e.g., Whole Sign or Placidus).

### 🧠 Core Astrological Engine
This section defines the mathematical and logical framework for the chart:
- `AstroTime`: Converts local time to Julian Day (JD) and handles UTC offsets.
- `AstroMath`: Provides utility functions for mapping longitudes to signs and nakshatras.
- `VedicAspectEngine`: Logic for calculating planetary sight (Graha Drishti).
- `FriendshipEngine`: Calculates natural and temporary relationships between planets (Panchadha Maitri).
- `DignityEngine`: Determines planetary strength (Exaltation, Moolatrikona, etc.).
- `VimshottariDashaEngine`: Projects time-based planetary cycles based on the Moon's position.

In [ ]:
# @title ⚙️ Birth Data Configuration

SUBJECT_NAME = "Steve Jobs" # @param {type:"string"}
B_YEAR = 1955 # @param {type:"integer"}
B_MONTH = 2 # @param {type:"integer"}
B_DAY = 24 # @param {type:"integer"}
B_TIME = "19:15" # @param {type:"string"}
B_TZ = "America/Los_Angeles" # @param {type:"string"}
GENDER = "Male" # @param ["Male", "Female"]

# @markdown **Geolocation Settings:**
BIRTH_PLACE = "San Francisco, CA" # @param {type:"string"}
AUTO_GEOLOCATE = True # @param {type:"boolean"}

# @markdown Manual coordinates (used if AUTO_GEOLOCATE is False or lookup fails):
LATITUDE = 37.7749 # @param {type:"number"}
LONGITUDE = -122.4194 # @param {type:"number"}

# @markdown ---
# @markdown ### Astrological Settings
# @markdown **Report Mode:** Controls what data is included in the OKF output.
# @markdown - `Tropical-Vedic`: Tropical Signs/Houses + Vedic Nakshatras/Dashas/Aspects.
REPORT_MODE = "Tropical-Vedic" # @param ["Tropical Focus", "Vedic Focus", "Everything", "Tropical-Vedic"]

HOUSE_SYSTEM = "W" # @param ["W", "P", "E"]
AYANAMSA = 36 # @param [36, 1, 3, 5]

# --- AUTO-COMPUTED EXPORT PATHS & METADATA ---
import os
import hashlib

# Perform automatic geolocation if requested
if AUTO_GEOLOCATE and BIRTH_PLACE:
    try:
        from geopy.geocoders import Nominatim
        geolocator = Nominatim(user_agent="astrology_engine")
        location = geolocator.geocode(BIRTH_PLACE)
        if location:
            LATITUDE = round(location.latitude, 4)
            LONGITUDE = round(location.longitude, 4)
            print(f"✅ Successfully geocoded '{BIRTH_PLACE}' to Lat: {LATITUDE}, Lon: {LONGITUDE}")
        else:
            print(f"⚠️ Could not geolocate '{BIRTH_PLACE}'. Using manual coordinates (Lat: {LATITUDE}, Lon: {LONGITUDE}).")
    except Exception as e:
        print(f"⚠️ Geolocation query failed: {e}. Using manual coordinates (Lat: {LATITUDE}, Lon: {LONGITUDE}).")
else:
    print(f"ℹ️ Auto-geolocation skipped. Using manual coordinates (Lat: {LATITUDE}, Lon: {LONGITUDE}).")

safe_subject = SUBJECT_NAME.lower().replace(" ", "_")
target_json_path = os.path.join("knowledge_base", safe_subject, f"{safe_subject}_astrology_profile.json")

# Generate unique fingerprint based on input parameters
fingerprint_input = f"{SUBJECT_NAME}_{B_YEAR}_{B_MONTH}_{B_DAY}_{B_TIME}_{LATITUDE}_{LONGITUDE}_{HOUSE_SYSTEM}_{AYANAMSA}"
fingerprint = hashlib.md5(fingerprint_input.encode('utf-8')).hexdigest()[:8]

subject_profile = {
    "name": SUBJECT_NAME,
    "gender": GENDER,
    "birth_data": {
        "date": f"{B_YEAR}-{B_MONTH:02d}-{B_DAY:02d}",
        "time": B_TIME,
        "timezone": B_TZ,
        "latitude": LATITUDE,
        "longitude": LONGITUDE
    }
}

metadata = {
    "engine": "Universal Astrological OKF Engine",
    "version": "2.0",
    "settings": {
        "report_mode": REPORT_MODE,
        "house_system": HOUSE_SYSTEM,
        "house_system_name": "Placidus" if HOUSE_SYSTEM == "P" else ("Equal" if HOUSE_SYSTEM == "E" else "Whole Sign"),
        "ayanamsa": AYANAMSA,
        "ayanamsa_name": "Galactic Center at 0 Mula (Wilhelm)" if AYANAMSA == 36 else (
            "Lahiri" if AYANAMSA == 1 else (
                "Raman" if AYANAMSA == 3 else (
                    "Fagan-Bradley" if AYANAMSA == 0 else (
                        "KP" if AYANAMSA == 5 else f"Custom ({AYANAMSA})"
                    )
                )
            )
        )
    }
}


# 🔱 Astrological Calculation Engines & Exporter

This section defines the unified calculation engines for computing divisional harmonics, planetary strength arrays, auspicious point distributions, and planetary yoga configurations. All engines dynamically adapt to the active `REPORT_MODE` (Tropical vs. Vedic coordinate frames).

### ⚙️ Engine Summary
1. **`AstroMath`**: Core utilities for sign, nakshatra, and house placements.
2. **`VargaEngine`**: Computes harmonic divisions (D1 through D60) based on active longitudes.
3. **`ShadbalaEngine`**: Calculates planetary strength across 6 categories (Sthana, Dig, Kala, Cheshta, Naisargika, Drik).
4. **`AshtakavargaEngine`**: Computes Bhinna (BAV) and Sarvashtakavarga (SAV) point matrices.
5. **`YogaEngine`**: Scans the birth chart for classic planetary yogas.
6. **`GocharaEngine`**: Evaluates transit alignments relative to the natal Moon position.
7. **`OKFConverter`**: Outputs structured, Obsidian-linked Markdown reports.

In [ ]:
# @title 🔱 Astrological Calculation Engines & Exporter
%load_ext autoreload
%autoreload 2

from astrology_engine import *


# 🚀 Calculation & Exporter Execution Pipeline

This cell executes the calculations sequentially:
1. Computes natal houses and instantiates planetary coordinates.
2. Caches house numbers on the planet objects directly to optimize friendship and divisional friendship loops.
3. Runs the complete Shadbala strength calculations, Ashtakavarga point grids, Yoga scanner, and Gochara transit checks.
4. Exports the structured results to JSON and runs the Obsidian converter.

> [!TIP]
> **Google Colab restoration**:
> To run this notebook in Google Colab, restore the Swiss Ephemeris path setting:
> `ephe_path = '/content/ephe'`

In [ ]:
try:
    # 1. Instantiate Core Timing and House Engine based on inputs
    epoch_time = AstroTime(B_YEAR, B_MONTH, B_DAY, B_TIME, B_TZ, sid_mode=AYANAMSA)
    local_sky = HouseEngine(epoch_time, LATITUDE, LONGITUDE, house_system=HOUSE_SYSTEM)
    my_birth_chart = ChartFactory.create_chart(epoch_time, mode=REPORT_MODE)

    # 2. Build Comprehensive Planet Data List and cache house numbers
    planets_data = []
    for name, obj in my_birth_chart.items():
        # Calculate signs once
        t_h = AstroMath.get_house_placement(obj.tropical_longitude, local_sky.houses, False)
        v_h = AstroMath.get_house_placement(obj.sidereal_longitude, local_sky.houses, True)
        
        # Cache calculated house numbers directly on the instances to eliminate redundant searches
        obj.tropical_house = t_h
        obj.vedic_house = v_h

    # 3. Calculate Complete Shadbala for the 7 classical planets (skipping Rahu/Ketu)
    for name, obj in my_birth_chart.items():
        is_sid = False if REPORT_MODE in ["Tropical-Vedic", "Tropical Focus"] else True
        house_num = obj.vedic_house if is_sid else obj.tropical_house
        shadbala = ShadbalaEngine.get_complete_shadbala(name, obj, my_birth_chart, local_sky.houses, house_num, REPORT_MODE)
        obj.shadbala = shadbala

    # 4. Formulate the planets payload array
    for name, obj in my_birth_chart.items():
        if REPORT_MODE == "Tropical-Vedic":
            planets_data.append({
                'name': name,
                'tropical_sign': obj.zodiac_sign,
                'tropical_house': obj.tropical_house,
                'tropical_degree_in_house': round(AstroMath.get_degree_in_house(obj.tropical_longitude, obj.tropical_house, local_sky.houses, False), 4),
                'vargas': obj.vargas,
                'dignity': obj.dignity,
                'shadbala': obj.shadbala,
                'sidereal_nakshatra': obj.nakshatra
            })
        else:
            planets_data.append({
                'name': name,
                'tropical_sign': obj.zodiac_sign,
                'tropical_house': obj.tropical_house,
                'tropical_degree_in_house': round(AstroMath.get_degree_in_house(obj.tropical_longitude, obj.tropical_house, local_sky.houses, False), 4),
                'vargas': obj.vargas,
                'sidereal_nakshatra': obj.nakshatra,
                'sidereal_sign': obj.sidereal_sign,
                'vedic_house': obj.vedic_house,
                'vedic_degree_in_house': round(AstroMath.get_degree_in_house(obj.sidereal_longitude, obj.vedic_house, local_sky.houses, True), 4),
                'dignity': obj.dignity,
                'tropical_dignity': obj.tropical_dignity,
                'sidereal_dignity': obj.sidereal_dignity,
                'shadbala': obj.shadbala
            })

    # 5. Calculate Ashtakavarga bindu allocations
    ashtakavarga = AshtakavargaEngine.calculate_ashtakavarga(my_birth_chart, local_sky, REPORT_MODE)

    # 6. Scan Yogas
    # 6. Scan Yogas dynamically from JSON rules file
    notebook_dir = os.path.abspath('')
    yogas_rules_path = os.path.join(notebook_dir, "yogas")
    yogas = YogaEngine.scan_yogas(my_birth_chart, local_sky, REPORT_MODE, yogas_rules_path, subject_profile)

    # 6.5. Scan Jaimini Karakas and Daily Panchanga
    jaimini_karakas = JaiminiEngine.get_karakas(my_birth_chart)
    panchanga = PanchangaEngine.calculate_panchanga(epoch_time, my_birth_chart, LATITUDE, LONGITUDE)

    # 7. Evaluate Gochara Transits (dynamic evaluation for current time)
    now_utc = datetime.now(timezone.utc)
    transit_time = AstroTime(now_utc.year, now_utc.month, now_utc.day, now_utc.hour + now_utc.minute/60.0, 0.0, sid_mode=AYANAMSA)
    transit_chart = ChartFactory.create_chart(transit_time, mode=REPORT_MODE)
    transits = GocharaEngine.evaluate_transits(my_birth_chart, transit_chart, REPORT_MODE)

    # 8. Assemble the unified JSON Chart Data
    if REPORT_MODE == "Tropical-Vedic":
        ascendant_data = {
            'tropical_sign': local_sky.tropical_ascendant_sign,
            'tropical_longitude': round(local_sky.tropical_ascendant, 4),
            'tropical_degree_in_house': round(AstroMath.get_degree_in_house(local_sky.tropical_ascendant, 1, local_sky.houses, False), 4)
        }
        # only tropical info for houses
        houses_data = {
            str(h): {"tropical_start": local_sky.houses[h]["tropical_start"]}
            for h in local_sky.houses
        }
    else:
        ascendant_data = {
            'tropical_sign': local_sky.tropical_ascendant_sign,
            'tropical_longitude': round(local_sky.tropical_ascendant, 4),
            'tropical_degree_in_house': round(AstroMath.get_degree_in_house(local_sky.tropical_ascendant, 1, local_sky.houses, False), 4),
            'sidereal_sign': local_sky.sidereal_ascendant_sign,
            'sidereal_longitude': round(local_sky.sidereal_ascendant, 4),
            'sidereal_degree_in_house': round(AstroMath.get_degree_in_house(local_sky.sidereal_ascendant, 1, local_sky.houses, True), 4)
        }
        houses_data = {str(h): local_sky.houses[h] for h in local_sky.houses}

    json_chart_data = {
        'ascendant': ascendant_data,
        'planets': planets_data,
        'houses': houses_data,
        'ashtakavarga': ashtakavarga,
        'yogas': yogas,
        'transits': transits,
        'vimshottari_dasha': VimshottariDashaEngine(epoch_time, my_birth_chart['Moon'].nakshatra, my_birth_chart['Moon'].sidereal_longitude).dasha_periods,
        'jaimini_karakas': jaimini_karakas,
        'panchanga': panchanga
    }

    # 9. Generate Planetary Compound Friendship Data
    json_friendship_data = {}
    classical = ["Sun", "Moon", "Mars", "Mercury", "Jupiter", "Venus", "Saturn"]
    for p1 in classical:
        json_friendship_data[p1] = {}
        for p2 in classical:
            if p1 == p2: continue
            is_sid = False if REPORT_MODE in ["Tropical-Vedic", "Tropical Focus"] else True
            h1 = my_birth_chart[p1].vedic_house if is_sid else my_birth_chart[p1].tropical_house
            h2 = my_birth_chart[p2].vedic_house if is_sid else my_birth_chart[p2].tropical_house
            json_friendship_data[p1][p2] = FriendshipEngine.get_panchadha_relationship(p1, p2, h1, h2)

    # 10. Assemble and Save payload JSON
    full_chart_export = {
        'metadata': metadata,
        'fingerprint': fingerprint,
        'subject': subject_profile,
        'astrological_chart': json_chart_data,
        'planetary_friendships': json_friendship_data
    }

    os.makedirs(os.path.dirname(target_json_path), exist_ok=True)
    with open(target_json_path, 'w', encoding='utf-8') as f:
        json.dump(full_chart_export, f, indent=4, ensure_ascii=False)

    # 10.5. Assemble and Save LLM-optimized compact payload JSON
    active_yogas = {
        name: {
            "description": data.get("description"),
            "results": data.get("results"),
            "details": data.get("details")
        }
        for name, data in json_chart_data["yogas"].items()
        if data.get("active")
    }

    from datetime import datetime
    today_str = datetime.now().strftime("%Y-%m-%d")
    current_dasha = {}
    for m in json_chart_data["vimshottari_dasha"]:
        if m["start"] <= today_str <= m["end"]:
            current_dasha["mahadasha"] = {"lord": m["lord"], "start": m["start"], "end": m["end"]}
            for a in m.get("antardashas", []):
                if a["start"] <= today_str <= a["end"]:
                    current_dasha["antardasha"] = {"lord": a["lord"], "start": a["start"], "end": a["end"]}
                    for p in a.get("pratyantardashas", []):
                        if p["start"] <= today_str <= p["end"]:
                            current_dasha["pratyantardasha"] = {"lord": p["lord"], "start": p["start"], "end": p["end"]}
                            break
                    break
            break

    llm_chart_data = {
        "ascendant": json_chart_data["ascendant"],
        "planets": planets_data,
        "houses": json_chart_data["houses"],
        "ashtakavarga_sav": json_chart_data["ashtakavarga"]["sav"],
        "active_yogas": active_yogas,
        "current_dasha": current_dasha,
        "transits": json_chart_data["transits"],
        "jaimini_karakas": json_chart_data["jaimini_karakas"],
        "panchanga": json_chart_data["panchanga"]
    }

    llm_payload = {
        "subject": subject_profile,
        "astrological_chart": llm_chart_data,
        "planetary_friendships": json_friendship_data
    }

    target_llm_json_path = os.path.join("knowledge_base", safe_subject, f"{safe_subject}_llm_payload.json")
    with open(target_llm_json_path, 'w', encoding='utf-8') as f:
        json.dump(llm_payload, f, indent=2, ensure_ascii=False)

    # 11. Generate HTML Report
    notebook_dir = os.path.abspath('')
    template_path = os.path.join(notebook_dir, "report_template.html")
    report_output_path = os.path.join(os.path.dirname(target_json_path), f"{safe_subject}_report.html")
    reporter = HTMLReporter(target_json_path, template_path)
    reporter.compile_report(report_output_path)

    print(f"✅ Calculation Pipeline Success. Data saved to: {target_json_path}")
    print(f"✅ HTML Report successfully generated at: {report_output_path}")
    print(f"Mode: {REPORT_MODE} | Divisional Data: {list(planets_data[0].get('vargas', {}).keys())}")

except Exception as e:
    print(f"❌ Execution Error: {e}")


In [ ]:
# 1. Verify Varga Content Integrity (D9, D10, D12, D60) from JSON
subject_dir = os.path.join('knowledge_base', safe_subject)

with open(target_json_path, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

first_planet_vargas = json_data['astrological_chart']['planets'][0].get('vargas', {})
vargas_checks = {v: (v in first_planet_vargas) for v in ['D9', 'D10', 'D12', 'D60']}

# Clean up any existing markdown files / 'charts' directory in output
charts_dir = os.path.join(subject_dir, 'charts')
if os.path.exists(charts_dir):
    import shutil
    shutil.rmtree(charts_dir)

# 2. Verify HTML Report Existence
report_file = os.path.join(subject_dir, f"{safe_subject}_report.html")
report_exists = os.path.exists(report_file) and os.path.getsize(report_file) > 0

# 3. Create Final Production Archive
final_zip_name = f"{safe_subject}_astrology_profile_final"
archive_path = shutil.make_archive(final_zip_name, 'zip', subject_dir)

print(f"--- Final Production Verification ---")
print(f"✅ Calculation Integrity Verified from JSON for: {SUBJECT_NAME}")
print(f"✅ Varga Integrity Checked: {vargas_checks}")
print(f"✅ HTML Report Generated & Verified: {report_exists}")
print(f"✅ Final Production Archive Created: {archive_path}")
print(f"\nThe engine is now production-ready with HTML dashboard output.")
